# Experiment 15 — Spatial-SAINT v2 Multi-Dataset Validation

**Objective:** Prove that the FYDP-III Spatial-SAINT pipeline (Transformer with Spatial Inter-Sample Attention + Stacking) generalizes beyond the original cHL CODEX dataset.

| Dataset | Disease | Imaging | Cells | Markers | Cell Types |
|---------|---------|---------|-------|---------|------------|
| cHL CODEX (Original) | Hodgkin Lymphoma | CODEX | 145K | 49 | 16 |
| CRC CODEX | Colorectal Cancer | CODEX | 258K | 56 | ~25 |
| cHL MIBI | Hodgkin Lymphoma | MIBI | 1.67M | 41 | 14 |


In [1]:
import os, random, time, warnings, math, copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler, SequentialSampler

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import f1_score, accuracy_score, classification_report
from sklearn.linear_model import LogisticRegression

warnings.filterwarnings('ignore', category=UserWarning)

SEED = 7325111

def set_seed(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

set_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch: {torch.__version__}')
print(f'Device: {DEVICE}')


PyTorch: 2.10.0+cu128
Device: cuda


## Spatial-SAINT Infrastructure (From Experiment 15)


In [2]:
class CODEXDataset(Dataset):
    def __init__(self, X_m, X_s, y=None):
        self.X_m = torch.tensor(X_m, dtype=torch.float32)
        self.X_s = torch.tensor(X_s, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long) if y is not None else None

    def __len__(self): return len(self.X_m)
    def __getitem__(self, idx):
        if self.y is not None: return self.X_m[idx], self.X_s[idx], self.y[idx]
        else: return self.X_m[idx], self.X_s[idx]

class FocalLoss(nn.Module):
    def __init__(self, alpha=1, gamma=2, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = self.alpha * (1-pt)**self.gamma * ce_loss
        if self.reduction == 'mean': return focal_loss.mean()
        elif self.reduction == 'sum': return focal_loss.sum()
        return focal_loss

class FeatureTokenizer(nn.Module):
    def __init__(self, n_features, d_model):
        super().__init__()
        self.W = nn.Parameter(torch.empty(n_features, d_model))
        self.b = nn.Parameter(torch.zeros(n_features, d_model))
        nn.init.kaiming_uniform_(self.W, a=math.sqrt(5))
    def forward(self, x): return x.unsqueeze(-1) * self.W + self.b

class SpatialEncoding(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(2, d_model), nn.GELU(), nn.Linear(d_model, d_model)
        )
    def forward(self, xy): return self.proj(xy)

class ColAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        h = self.norm(x)
        h, _ = self.attn(h, h, h)
        return x + self.drop(h)

class RowAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        B, S, D = x.shape
        x_t  = x.permute(1, 0, 2)
        h    = self.norm(x_t)
        h, _ = self.attn(h, h, h)
        x_t  = x_t + self.drop(h)
        return x_t.permute(1, 0, 2)

class FFNBlock(nn.Module):
    def __init__(self, d_model, mult=4, dropout=0.1):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.ffn  = nn.Sequential(
            nn.Linear(d_model, d_model * mult), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_model * mult, d_model), nn.Dropout(dropout)
        )
    def forward(self, x): return x + self.ffn(self.norm(x))

class SpatialSAINT(nn.Module):
    def __init__(self, n_features, num_classes, d_model=128, n_heads=8, n_layers=2, dropout=0.15):
        super().__init__()
        self.tokenizer = FeatureTokenizer(n_features, d_model)
        self.spat_enc  = SpatialEncoding(d_model)
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model))

        self.layers = nn.ModuleList()
        for _ in range(n_layers):
            self.layers.append(nn.ModuleDict({
                'col_attn': ColAttention(d_model, n_heads, dropout),
                'row_attn': RowAttention(d_model, n_heads, dropout),
                'ffn': FFNBlock(d_model, mult=4, dropout=dropout)
            }))

        self.head = nn.Sequential(
            nn.LayerNorm(d_model), nn.Linear(d_model, d_model // 2),
            nn.GELU(), nn.Dropout(dropout), nn.Linear(d_model // 2, num_classes)
        )

    def forward(self, X_m, X_s):
        B = X_m.shape[0]
        tokens = self.tokenizer(X_m)
        spatial_ctx = self.spat_enc(X_s).unsqueeze(1)
        cls = self.cls_token.expand(B, -1, -1) + spatial_ctx
        tokens = torch.cat([cls, tokens], dim=1)

        for layer in self.layers:
            tokens = layer['col_attn'](tokens)
            tokens = layer['row_attn'](tokens)
            tokens = layer['ffn'](tokens)

        return self.head(tokens[:, 0])

def predict_proba(model, X_m, X_s, batch_size=256):
    model.eval()
    ds = CODEXDataset(X_m, X_s)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False)
    all_probs = []
    with torch.no_grad():
        for xb_m, xb_s in loader:
            logits = model(xb_m.to(DEVICE), xb_s.to(DEVICE))
            probs = F.softmax(logits, dim=-1)
            all_probs.append(probs.cpu().numpy())
    return np.concatenate(all_probs, axis=0)


## Reusable Pipeline Function


In [3]:
def train_fold(X_m_tr, X_s_tr, y_tr, X_m_va, X_s_va, y_va, n_features, n_classes, max_epochs=200, patience=40, lr=3e-4):
    ds_tr = CODEXDataset(X_m_tr, X_s_tr, y_tr)
    ds_va = CODEXDataset(X_m_va, X_s_va, y_va)

    counts = np.bincount(y_tr)
    weights = 1.0 / counts[y_tr]
    sampler = WeightedRandomSampler(weights, len(weights), replacement=True)

    tr_loader = DataLoader(ds_tr, batch_size=256, sampler=sampler, drop_last=True)
    va_loader = DataLoader(ds_va, batch_size=256, shuffle=False)

    model = SpatialSAINT(n_features=n_features, num_classes=n_classes, d_model=128, n_heads=8, n_layers=2, dropout=0.15).to(DEVICE)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=50, T_mult=2)
    criterion = FocalLoss(alpha=1, gamma=2, reduction='mean')
    scaler = torch.cuda.amp.GradScaler()

    best_f1, patience_ctr, best_state = 0.0, 0, None
    for epoch in range(max_epochs):
        model.train()
        for xb_m, xb_s, yb in tr_loader:
            xb_m, xb_s, yb = xb_m.to(DEVICE), xb_s.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            with torch.cuda.amp.autocast():
                logits = model(xb_m, xb_s)
                loss = criterion(logits, yb)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
        scheduler.step()

        # Eval
        probs = predict_proba(model, X_m_va, X_s_va)
        preds = np.argmax(probs, axis=1)
        va_f1 = f1_score(y_va, preds, average='weighted')

        if va_f1 > best_f1:
            best_f1 = va_f1
            best_state = copy.deepcopy(model.state_dict())
            patience_ctr = 0
        else:
            patience_ctr += 1
        if patience_ctr > patience:
            break

    model.load_state_dict(best_state)
    return model, best_f1

def run_saint_pipeline(X_m, X_s, y, class_names, dataset_name, max_cells=None):
    """Runs 5-Fold CV + Stacking for Spatial-SAINT."""
    set_seed(SEED)
    NUM_FEATURES = X_m.shape[1]
    NUM_CLASSES = len(class_names)

    if max_cells and len(X_m) > max_cells:
        print(f"Subsampling {len(X_m)} -> {max_cells} cells (stratified)...")
        idx, _ = train_test_split(np.arange(len(X_m)), train_size=max_cells, random_state=SEED, stratify=y)
        X_m, X_s, y = X_m[idx], X_s[idx], y[idx]

    print(f'\n{"="*70}')
    print(f'RUNNING SPATIAL-SAINT 5-FOLD CV ON: {dataset_name}')
    print(f'{"="*70}')

    # Standard Scale markers, MinMax Scale spatial coordinates
    scaler_m = StandardScaler()
    X_m_sc = scaler_m.fit_transform(X_m)
    
    from sklearn.preprocessing import MinMaxScaler
    scaler_s = MinMaxScaler()
    X_s_sc = scaler_s.fit_transform(X_s)

    # Split a holdout test set (20%)
    X_tr_m, X_te_m, X_tr_s, X_te_s, y_tr, y_te = train_test_split(
        X_m_sc, X_s_sc, y, test_size=0.2, random_state=SEED, stratify=y)

    # 5-Fold CV on Train
    N_FOLDS = 5
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    
    oof_probs = np.zeros((len(y_tr), NUM_CLASSES))
    test_probs = np.zeros((len(y_te), NUM_CLASSES))
    fold_f1s = []

    for fold, (tr_idx, oof_idx) in enumerate(skf.split(X_tr_m, y_tr)):
        print(f'\n--- FOLD {fold+1}/{N_FOLDS} ---')
        X_m_f_tr, X_m_f_va = X_tr_m[tr_idx], X_tr_m[oof_idx]
        X_s_f_tr, X_s_f_va = X_tr_s[tr_idx], X_tr_s[oof_idx]
        y_f_tr, y_f_va = y_tr[tr_idx], y_tr[oof_idx]

        model, fold_f1 = train_fold(
            X_m_f_tr, X_s_f_tr, y_f_tr, X_m_f_va, X_s_f_va, y_f_va,
            n_features=NUM_FEATURES, n_classes=NUM_CLASSES)
        fold_f1s.append(fold_f1)
        
        oof_probs[oof_idx] = predict_proba(model, X_m_f_va, X_s_f_va)
        test_probs += predict_proba(model, X_te_m, X_te_s) / N_FOLDS
        print(f'Fold {fold+1} F1: {fold_f1:.4f}')

    mean_cv_f1 = np.mean(fold_f1s)
    print(f'\nCV Mean F1: {mean_cv_f1:.4f} ± {np.std(fold_f1s):.4f}')

    # Stacking (Meta-Learner)
    print('\nTraining Meta-Learner (Logistic Regression) on OOF predictions...')
    meta = LogisticRegression(max_iter=1000, random_state=SEED)
    meta.fit(oof_probs, y_tr)
    
    stack_preds = meta.predict(test_probs)
    stack_f1 = f1_score(y_te, stack_preds, average='weighted')
    print(f'\nTest Set Stacking F1: {stack_f1:.4f}')

    return {'dataset': dataset_name, 'cv_mean_f1': mean_cv_f1, 
            'stack_f1': stack_f1, 'num_cells': len(X_m), 
            'num_features': NUM_FEATURES, 'num_classes': NUM_CLASSES}


---
## Dataset 1: CRC CODEX (Colorectal Cancer)
Same imaging technology (CODEX), different disease → proves **cross-tissue generalization**.


In [4]:
print("Loading CRC CODEX dataset...")
df_crc = pd.read_csv("/kaggle/input/datasets/mdkhademulislamnahin/crc-clusters/CRC_clusters_neighborhoods_markers.csv")
print(f"Raw: {len(df_crc)} cells")

# Drop noisy / ambiguous classes
drop_classes = ['dirt', 'undefined', 'immune cells / vasculature', 'tumor cells / immune cells']
df_crc = df_crc[~df_crc['ClusterName'].isin(drop_classes)]
print(f"After cleanup: {len(df_crc)} cells, {df_crc['ClusterName'].nunique()} cell types")

meta_cols = ['Unnamed: 0','CellID','ClusterID','EventID','File Name','Region',
    'TMA_AB','TMA_12','Index in File','groups','patients','spots',
    'cell_id:cell_id','tile_nr:tile_nr','X:X','Y:Y',
    'X_withinTile:X_withinTile','Y_withinTile:Y_withinTile','Z:Z',
    'size:size','HOECHST1:Cyc_1_ch_1','DRAQ5:Cyc_23_ch_4',
    'Profile_Homogeneity:Fiter1','ClusterSize','ClusterName',
    'neighborhood10','neighborhood number final','neighborhood name']
binary_cols = [c for c in df_crc.columns if '+' in c]
exclude = set(meta_cols + binary_cols)
marker_cols_crc = [c for c in df_crc.columns if c not in exclude]

le_crc = LabelEncoder()
y_crc = le_crc.fit_transform(df_crc['ClusterName'].values)
class_names_crc = le_crc.classes_.tolist()
X_m_crc = df_crc[marker_cols_crc].values.astype(np.float32)
X_s_crc = df_crc[['X:X', 'Y:Y']].values.astype(np.float32) # Spatial Coords
print(f"Ready: X_m={X_m_crc.shape}, X_s={X_s_crc.shape}, Classes={len(class_names_crc)}")


Loading CRC CODEX dataset...
Raw: 258385 cells
After cleanup: 240554 cells, 25 cell types
Ready: X_m=(240554, 56), X_s=(240554, 2), Classes=25


### Run Pipeline on CRC CODEX


In [5]:
crc_results = run_saint_pipeline(
    X_m_crc, X_s_crc, y_crc, class_names_crc,
    dataset_name="CRC CODEX (Colorectal Cancer)")



RUNNING SPATIAL-SAINT 5-FOLD CV ON: CRC CODEX (Colorectal Cancer)

--- FOLD 1/5 ---


/tmp/ipykernel_23/3528324712.py:16: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
/tmp/ipykernel_23/3528324712.py:24: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/tmp/ipykernel_23/3528324712.py:24: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/tmp/ipykernel_23/3528324712.py:24: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/tmp/ipykernel_23/3528324712.py:24: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/tmp/ipykernel_23/35283247

Fold 1 F1: 0.8420

--- FOLD 2/5 ---


/tmp/ipykernel_23/3528324712.py:16: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
/tmp/ipykernel_23/3528324712.py:24: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/tmp/ipykernel_23/3528324712.py:24: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/tmp/ipykernel_23/3528324712.py:24: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/tmp/ipykernel_23/3528324712.py:24: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/tmp/ipykernel_23/35283247

Fold 2 F1: 0.8568

--- FOLD 3/5 ---


...

Fold 3 F1: 0.8414

--- FOLD 4/5 ---


/tmp/ipykernel_23/3528324712.py:16: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
/tmp/ipykernel_23/3528324712.py:24: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/tmp/ipykernel_23/3528324712.py:24: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/tmp/ipykernel_23/3528324712.py:24: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/tmp/ipykernel_23/3528324712.py:24: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/tmp/ipykernel_23/35283247

Fold 4 F1: 0.8469

--- FOLD 5/5 ---


/tmp/ipykernel_23/3528324712.py:16: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
/tmp/ipykernel_23/3528324712.py:24: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/tmp/ipykernel_23/3528324712.py:24: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/tmp/ipykernel_23/3528324712.py:24: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/tmp/ipykernel_23/3528324712.py:24: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/tmp/ipykernel_23/35283247

Fold 5 F1: 0.8557

CV Mean F1: 0.8486 ± 0.0066

Training Meta-Learner (Logistic Regression) on OOF predictions...

Test Set Stacking F1: 0.8799


---
## Dataset 2: cHL MIBI (Hodgkin Lymphoma — MIBI)
Same disease, different imaging technology → proves **cross-platform generalization**.


In [6]:
print("Loading cHL MIBI dataset...")
df_mibi = pd.read_csv("/kaggle/input/datasets/mdkhademulislamnahin/chl1-mibi/cHL1_MIBI.csv")
print(f"Raw: {len(df_mibi)} cells")

mibi_meta = ['cellLabel','Annotation','centroidX','centroidY','cellSize','identifier']
marker_cols_mibi = [c for c in df_mibi.columns if c not in mibi_meta]

le_mibi = LabelEncoder()
y_mibi = le_mibi.fit_transform(df_mibi['Annotation'].values)
class_names_mibi = le_mibi.classes_.tolist()
X_m_mibi = df_mibi[marker_cols_mibi].values.astype(np.float32)
X_s_mibi = df_mibi[['centroidX', 'centroidY']].values.astype(np.float32) # Spatial Coords
print(f"Ready: X_m={X_m_mibi.shape}, X_s={X_s_mibi.shape}, Classes={len(class_names_mibi)}")


Loading cHL MIBI dataset...
Raw: 1669853 cells
Ready: X_m=(1669853, 41), X_s=(1669853, 2), Classes=14


### Run Pipeline on cHL MIBI
> Subsampling to 100K cells for 5-Fold CV (very slow otherwise).


In [7]:
mibi_results = run_saint_pipeline(
    X_m_mibi, X_s_mibi, y_mibi, class_names_mibi,
    dataset_name="cHL MIBI (Hodgkin Lymphoma)", max_cells=100000)


Subsampling 1669853 -> 100000 cells (stratified)...

RUNNING SPATIAL-SAINT 5-FOLD CV ON: cHL MIBI (Hodgkin Lymphoma)

--- FOLD 1/5 ---


/tmp/ipykernel_23/3528324712.py:16: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
/tmp/ipykernel_23/3528324712.py:24: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/tmp/ipykernel_23/3528324712.py:24: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/tmp/ipykernel_23/3528324712.py:24: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/tmp/ipykernel_23/3528324712.py:24: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/tmp/ipykernel_23/35283247

Fold 1 F1: 0.8812

--- FOLD 2/5 ---


/tmp/ipykernel_23/3528324712.py:16: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
/tmp/ipykernel_23/3528324712.py:24: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/tmp/ipykernel_23/3528324712.py:24: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/tmp/ipykernel_23/3528324712.py:24: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/tmp/ipykernel_23/3528324712.py:24: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/tmp/ipykernel_23/35283247

Fold 2 F1: 0.8820

--- FOLD 3/5 ---


...

Fold 3 F1: 0.8817

--- FOLD 4/5 ---


/tmp/ipykernel_23/3528324712.py:16: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
/tmp/ipykernel_23/3528324712.py:24: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/tmp/ipykernel_23/3528324712.py:24: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/tmp/ipykernel_23/3528324712.py:24: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/tmp/ipykernel_23/3528324712.py:24: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/tmp/ipykernel_23/35283247

Fold 4 F1: 0.8854

--- FOLD 5/5 ---


/tmp/ipykernel_23/3528324712.py:16: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
/tmp/ipykernel_23/3528324712.py:24: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/tmp/ipykernel_23/3528324712.py:24: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/tmp/ipykernel_23/3528324712.py:24: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/tmp/ipykernel_23/3528324712.py:24: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/tmp/ipykernel_23/35283247

Fold 5 F1: 0.8836

CV Mean F1: 0.8828 ± 0.0015

Training Meta-Learner (Logistic Regression) on OOF predictions...

Test Set Stacking F1: 0.9120


---
## Cross-Dataset Comparison Table


In [8]:
orig = {'dataset':'cHL CODEX (Original)', 'cv_mean_f1':0.8715,
    'stack_f1': 0.8787, 'num_cells':145161, 'num_features':50, 'num_classes':16}

all_r = [orig, crc_results, mibi_results]

print('\n' + '='*90)
print('CROSS-DATASET GENERALIZATION RESULTS: SPATIAL-SAINT v2')
print('='*90)
hdr = f'{"Metric":<25}'
for r in all_r: hdr += f' | {r["dataset"][:22]:>22}'
print(hdr)
print('-'*90)

for label, key, fmt in [('Cells','num_cells','{:,.0f}'),('Features','num_features','{:d}'),
    ('Cell Types','num_classes','{:d}'),('CV Mean F1','cv_mean_f1','{:.4f}'),
    ('Stacking F1','stack_f1','{:.4f}')]:
    row = f'  {label:<23}'
    for r in all_r: row += f' | {fmt.format(r[key]):>22}'
    print(row)

print(f'\n{"="*90}')
print('KEY FINDINGS:')
print('\n  CONFIRMED: Spatial-SAINT pipeline correctly executes on novel tissues and modalities.')
print('='*90)



CROSS-DATASET GENERALIZATION RESULTS: SPATIAL-SAINT v2
Metric                    |   cHL CODEX (Original) | CRC CODEX (Colorectal  | cHL MIBI (Hodgkin Lymp
------------------------------------------------------------------------------------------
  Cells                   |                145,161 |                240,554 |                100,000
  Features                |                     50 |                     56 |                     41
  Cell Types              |                     16 |                     25 |                     14
  CV Mean F1              |                 0.8715 |                 0.8486 |                 0.8828
  Stacking F1             |                 0.8787 |                 0.8799 |                 0.9120

KEY FINDINGS:

  CONFIRMED: Spatial-SAINT pipeline correctly executes on novel tissues and modalities.
